# 13 · Current Grading Pipeline (mirrors the running server)

This notebook imports the **exact** modules the Chrome-extension grading server runs
(`../grading_server/`), so you can reproduce the live `/grade` output here and
**test for discrepancies** between the notebook and the deployed pipeline.

**Pipeline (Stage A → Stage B):**
1. **YOLO OBB** detect the card → `_order_corners` → `refine_quad_to_edges` → `adaptive_padding`
2. **Perspective warp** to 630×880 (`_warp_card`) + analytical `card_boundary`
3. **Palette centering** hint (`analytical_centering`) — Pokémon border-colour HSV
4. **Multi-crop** (full card + 4 corner zooms) → **Claude `claude-sonnet-4-5`** → pillar JSON
5. **Geometric centering** recomputed from Claude's `content_region`
6. **Stage-B aggregator** (`aggregator.py`, linear `grade_model.json`) → overall grade
7. **Front + Back** → worst-side-per-pillar combine
8. **eBay comps + ROI** (`comps.py`)

**Requirements:** `ANTHROPIC_API_KEY` set, YOLO weights present (`YOLO_WEIGHTS`), and the
deps from `grading_server/requirements.txt` (ultralytics, opencv-python, anthropic, …).


In [ ]:
import os, sys, json
from pathlib import Path

# Import the SAME modules the server runs.
GRADING_SERVER = str((Path.cwd().parent / "grading_server").resolve())
if GRADING_SERVER not in sys.path:
    sys.path.insert(0, GRADING_SERVER)

# Configure before importing grader (it reads env at import time).
os.environ.setdefault("CLAUDE_MODEL", "claude-sonnet-4-5")
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."         # <- set if not already in env
# os.environ["YOLO_WEIGHTS"] = "/path/to/best.pt"        # <- set if not the default

import cv2
import numpy as np
import matplotlib.pyplot as plt

from grader import detect_and_grade, MODEL, YOLO_WEIGHTS
from aggregator import aggregate_overall, merge_pillars, psa_label, PILLARS
from comps import compute_economics

print("grading_server :", GRADING_SERVER)
print("model          :", MODEL)
print("yolo weights   :", YOLO_WEIGHTS, "(exists:", Path(YOLO_WEIGHTS).exists(), ")")
print("api key set    :", bool(os.environ.get("ANTHROPIC_API_KEY")))


## 1 · Grade a single side (Stage A)

`detect_and_grade(img_bgr)` runs the full Stage-A pipeline and returns the per-pillar
scores, `overall_score`, `psa_equivalent`, plus debug fields (`_card_boundary`,
`centering.content_region`, `_warped_jpeg_b64`, `_corner_crops_b64`, `_quad_padded`…).

In [ ]:
FRONT = "image0_front.jpeg"   # sample image shipped in notebooks/
img = cv2.imread(FRONT)
assert img is not None, f"cannot read {FRONT}"

front = detect_and_grade(img)   # ⚠ one Claude call

def pillars(r):
    return {k: (r.get(k) or {}).get("score") for k in PILLARS}

print("overall :", front.get("overall_score"), "/", front.get("psa_equivalent"))
print("pillars :", pillars(front))
print("centering:", front.get("centering", {}).get("left_right"),
      "/", front.get("centering", {}).get("top_bottom"))


In [ ]:
# Visualize the centering audit: green = card_boundary, gold = content_region.
import base64
warped = cv2.imdecode(np.frombuffer(base64.b64decode(front["_warped_jpeg_b64"]), np.uint8),
                      cv2.IMREAD_COLOR)
H, W = warped.shape[:2]
cb = front["_card_boundary"]                       # [x1,y1,x2,y2] (0..1)
cr = front.get("centering", {}).get("content_region")  # {x1,y1,x2,y2} (0..1)

fig, ax = plt.subplots(figsize=(5, 7))
ax.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)); ax.axis("off")
import matplotlib.patches as patches
if cb:
    ax.add_patch(patches.Rectangle((cb[0]*W, cb[1]*H), (cb[2]-cb[0])*W, (cb[3]-cb[1])*H,
                 lw=2.5, edgecolor="lime", facecolor="none"))
if cr:
    ax.add_patch(patches.Rectangle((cr["x1"]*W, cr["y1"]*H), (cr["x2"]-cr["x1"])*W, (cr["y2"]-cr["y1"])*H,
                 lw=2, edgecolor="gold", ls="--", facecolor="none"))
ax.set_title(f"Centering {front['centering'].get('left_right')} / {front['centering'].get('top_bottom')}")
plt.show()


## 2 · Front + Back → combined grade (Stage B)

The server grades each side independently, then combines **worst-side per pillar**
(`merge_pillars`) and runs the linear **Stage-B aggregator** (`aggregate_overall`,
weights from `grade_model.json`). This is exactly what `/grade` returns as `_combined`.

In [ ]:
BACK = "image0_back.jpeg"
back_img = cv2.imread(BACK)
back = detect_and_grade(back_img) if back_img is not None else None   # ⚠ one Claude call

combined = merge_pillars(front, back) if back else pillars(front)
overall  = aggregate_overall(combined)
print("front    :", pillars(front), "->", front.get("overall_score"))
if back:
    print("back     :", pillars(back),  "->", back.get("overall_score"))
print("combined :", combined, "-> overall", overall, psa_label(overall))


## 3 · Discrepancy check vs the live server

Posts the same front+back to the running server (`localhost:8000/grade`) and compares
its pillar scores to what we computed locally above. They should match — if they don't,
the notebook and the deployed pipeline have drifted.

In [ ]:
import requests
try:
    files = {"image": open(FRONT, "rb")}
    if back is not None:
        files["image_back"] = open(BACK, "rb")
    resp = requests.post("http://127.0.0.1:8000/grade", files=files,
                         data={"title": "Charizard Base Set", "price": "40", "shipping": "4"},
                         timeout=200).json()
    print("server front pillars:", {k: (resp.get(k) or {}).get("score") for k in PILLARS})
    print("server _combined    :", resp.get("_combined"))
    print("server decision     :", resp.get("decision"), "| comps:", resp.get("_comps_source"))
    # Compare local front pillars vs server front pillars
    local_f, server_f = pillars(front), {k: (resp.get(k) or {}).get("score") for k in PILLARS}
    print("\nMATCH (front pillars):", local_f == server_f)
    if local_f != server_f:
        print("  local :", local_f)
        print("  server:", server_f, "  <- DISCREPANCY (note: separate Claude calls vary slightly)")
except Exception as e:
    print("Server not reachable or error:", e)
    print("Start it:  cd ../grading_server && ./start.sh")


## 4 · Economics (eBay comps + ROI)

`compute_economics(title, price, shipping, overall_score)` searches eBay sold comps
(falls back to active listings), runs the ROI model, and returns a buy/maybe/skip
decision. Returns `unknown` ("NO DATA") when the eBay Finding API quota is exhausted.

In [ ]:
econ = compute_economics(title="Charizard Base Set", price=40, shipping=4,
                         overall_score=overall or front.get("overall_score") or 0)
print(json.dumps(econ, indent=2))
